# 🍩 Glaze & Classify — Interactive Explorer

Translate and classify bakery products with **Cortex AI**. Enter any product name in any language and watch the pipeline translate and classify it.

This notebook complements the [Streamlit dashboard](https://docs.snowflake.com/en/user-guide/ui-snowsight/streamlit-about) (metrics & charts). When you're done exploring, two AI tools pick up where this notebook leaves off:

- **"What are the results?"** → [Intelligence Agent](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents) — open from **AI & ML > Snowflake Intelligence**
- **"How do I improve?"** → [Cortex Code](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code) — open this repo in a **Workspace**

---

In [ ]:
USE SCHEMA SNOWFLAKE_EXAMPLE.GLAZE_AND_CLASSIFY;
USE WAREHOUSE SFE_GLAZE_AND_CLASSIFY_WH;

## Live Classify

Enter any product name — in **any language** — and run the cell to translate and classify it with Cortex AI.

Try: `チョコレート グレーズド リング`, `Croissant aux amandes`, `Dona Glaseada Original`

In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

# ── Change this value and re-run the cell ──
product_name = "チョコレート グレーズド リング"

# Step 1: Translate to English
translated = session.sql(
    "SELECT AI_TRANSLATE(?, '', 'en') AS translated",
    params=[product_name],
).collect()[0]["TRANSLATED"]

print(f"Translated: {product_name}  →  {translated}")

# Step 2: Classify with structured output
# Uses raw SQL (no params) because Snowpark parameterization
# corrupts the TYPE OBJECT expression. ::VARCHAR avoids OBJECT
# type in the result DataFrame.
prompt = (
    "Classify this bakery product into exactly one category and subcategory. "
    "Categories: Glazed, Frosted, Filled, Cake, Specialty, Seasonal, "
    "Beverages, Merchandise.\\n\\n"
    f"Product: {translated}"
)

result = session.sql(
    "SELECT AI_COMPLETE("
    "  model => 'claude-haiku-4-5',"
    f"  prompt => $${prompt}$$,"
    "  response_format => TYPE OBJECT(category VARCHAR, subcategory VARCHAR)"
    ")::VARCHAR AS result"
).collect()[0]["RESULT"]

parsed = json.loads(result)
print(f"Category:    {parsed['category']}")
print(f"Subcategory: {parsed['subcategory']}")

### Raw SQL version

The same operation as pure SQL — useful for embedding in pipelines.

In [ ]:
SELECT
    'チョコレート グレーズド リング'                    AS original,
    AI_TRANSLATE('チョコレート グレーズド リング', '', 'en') AS translated,
    AI_COMPLETE(
        model => 'claude-haiku-4-5',
        prompt => CONCAT(
            'Classify this bakery product into exactly one category and subcategory. ',
            'Categories: Glazed, Frosted, Filled, Cake, Specialty, Seasonal, Beverages, Merchandise.\n\n',
            'Product: ', AI_TRANSLATE('チョコレート グレーズド リング', '', 'en')
        ),
        response_format => {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'category': {'type': 'string'},
                    'subcategory': {'type': 'string'}
                },
                'required': ['category', 'subcategory']
            }
        }
    ) AS classification;

---

## What's next?

### "What are the results?" → Intelligence Agent

Open the **Glaze & Classify Assistant** at **AI & ML > Snowflake Intelligence** in Snowsight. Ask questions about the data:
- *What is the overall accuracy of each classification approach?*
- *Which products are misclassified by traditional SQL?*
- *How does accuracy compare across languages?*
- *Show me low-confidence predictions from the robust pipeline*

### "How do I improve?" → Cortex Code

Open this repository in a **Workspace** (**Projects > Workspaces**) and use [Cortex Code](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code) to ask about optimizing the pipeline:
- *How can I improve the robust classification prompt?*
- *What would it cost to switch from llama3.3-70b to a larger model?*
- *Suggest a better response_format schema for edge cases*
- *How do I add a new market to the pipeline?*